# RHI Live Runtime v27 — Input Induction Recursive Solver

Δ **Purpose:** convert the prototype into a real model-origin recursive solver.

Core lock:

$$
\boxed{
\text{Do not recurse on answers. Recurse on residue by mutating the contract.}
}
$$

Architecture:

$$
Q_{\text{raw}}
\rightarrow
\text{InputInductionPacket}
\rightarrow
C_0
\rightarrow
B_i
\rightarrow
A_i
\rightarrow
\Omega_t
\rightarrow
\Delta C_t
\rightarrow
C_{t+1}
\rightarrow
\Psi/\Omega/\bot
$$

This notebook corrects the prototype issues:

1. $\Psi$ requires **model-origin** answer.
2. `polysemy_check` is wired into branch rejection.
3. Recursion mutates the **contract**, not the prompt text.
4. Residue produces structured contract patches.
5. H is not part of runtime scoring.
6. Output is exactly two files:

```text
rhi_v27_<run_id>_bundle.json
rhi_v27_<run_id>_summary.csv
```


In [1]:

from __future__ import annotations

import os, re, sys, json, math, uuid, time, random, traceback, subprocess, importlib, hashlib
from dataclasses import dataclass, asdict, field
from enum import Enum
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

ROOT = Path.cwd()
OUT_DIR = ROOT / "rhi_v27_outputs"
OUT_DIR.mkdir(exist_ok=True)

RUN_ID = "rhi_v27_" + uuid.uuid4().hex[:10]
SEED = 27
random.seed(SEED)
np.random.seed(SEED)

MODEL_ID_OR_PATH = os.environ.get("RHI_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
LOAD_REAL_MODEL = True
REQUIRE_MODEL_FOR_PSI = True
AUTO_INSTALL_MISSING_DEPS = True

MAX_DEPTH = 3
MAX_NEW_TOKENS_BRANCH = 220
MAX_NEW_TOKENS_INDUCTION = 260
MAX_NEW_TOKENS_PATCH = 220
MAX_NEW_TOKENS_SHAPER = 160

TEMPERATURE_INDUCTION = 0.25
TEMPERATURE_BRANCH = 0.45
TEMPERATURE_PATCH = 0.25
TEMPERATURE_SHAPER = 0.25

BRANCH_ROLES = ["construct", "verify", "counter", "repair", "explore"]

RUN_PROMPT_LIMIT = 36
SHAPER_ENABLED = True

print("RHI v27 — Input Induction Recursive Solver")
print("RUN_ID:", RUN_ID)
print("MODEL:", MODEL_ID_OR_PATH)
print("OUT_DIR:", OUT_DIR)


RHI v27 — Input Induction Recursive Solver
RUN_ID: rhi_v27_3c42551ef0
MODEL: Qwen/Qwen2.5-1.5B-Instruct
OUT_DIR: D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v27_outputs


## 1. Dependency and Model Setup

Valid collapse requires real model output. If the model is not ready, the run stops or returns $\Omega_{\text{model}}$; it never fabricates $\Psi$ from fallback text.


In [2]:

def _module_available(module_name: str) -> bool:
    try:
        return importlib.util.find_spec(module_name) is not None
    except ModuleNotFoundError:
        return False
    except Exception:
        return False

def ensure_runtime_dependencies() -> Dict[str, Any]:
    status = {
        "checked": True,
        "attempted_install": False,
        "missing_before": [],
        "missing_after": [],
        "errors": []
    }
    required = [
        ("torch", "torch"),
        ("transformers", "transformers"),
        ("sentencepiece", "sentencepiece"),
        ("google.protobuf", "protobuf"),
    ]
    for module_name, pip_name in required:
        if not _module_available(module_name):
            status["missing_before"].append(pip_name)

    if status["missing_before"] and AUTO_INSTALL_MISSING_DEPS:
        status["attempted_install"] = True
        try:
            subprocess.run(
                [sys.executable, "-m", "pip", "install", *sorted(set(status["missing_before"]))],
                check=True
            )
            importlib.invalidate_caches()
        except Exception as e:
            status["errors"].append(repr(e))

    for module_name, pip_name in required:
        if not _module_available(module_name):
            status["missing_after"].append(pip_name)

    return status

DEPENDENCY_STATUS = ensure_runtime_dependencies()

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def get_device_info() -> Dict[str, Any]:
    info = {
        "torch_version": torch.__version__,
        "cuda_available": bool(torch.cuda.is_available()),
        "device_count": int(torch.cuda.device_count()) if torch.cuda.is_available() else 0,
        "cuda_version": getattr(torch.version, "cuda", None),
    }
    if torch.cuda.is_available():
        info["gpu_name"] = torch.cuda.get_device_name(0)
    return info

DEVICE_INFO = get_device_info()

tokenizer = None
model = None
MODEL_READY = False
MODEL_GENERATION_READY = False
MODEL_ERROR = None
SMOKE_TEXT = None

def load_model():
    global tokenizer, model, MODEL_READY, MODEL_ERROR
    if not LOAD_REAL_MODEL:
        MODEL_READY = False
        MODEL_ERROR = "LOAD_REAL_MODEL=False"
        return

    try:
        print("Loading model:", MODEL_ID_OR_PATH)
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID_OR_PATH)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID_OR_PATH,
            dtype=dtype,
            device_map="auto" if torch.cuda.is_available() else None,
        )
        if not torch.cuda.is_available():
            model = model.to("cpu")
        model.eval()
        MODEL_READY = True
        print("MODEL_READY:", MODEL_READY, "DEVICE:", next(model.parameters()).device)
    except Exception:
        MODEL_ERROR = traceback.format_exc()
        MODEL_READY = False
        print("MODEL LOAD ERROR")
        print(MODEL_ERROR)

def model_smoke_test():
    global MODEL_GENERATION_READY, SMOKE_TEXT, MODEL_ERROR
    if not MODEL_READY:
        MODEL_GENERATION_READY = False
        return

    try:
        messages = [{"role": "user", "content": "Reply with READY only."}]
        rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(rendered, return_tensors="pt").to(next(model.parameters()).device)

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=8,
                do_sample=False,
                return_dict_in_generate=True,
                pad_token_id=tokenizer.eos_token_id,
            )

        gen = out.sequences[0][inputs.input_ids.shape[1]:]
        SMOKE_TEXT = tokenizer.decode(gen, skip_special_tokens=True).strip()
        MODEL_GENERATION_READY = bool(SMOKE_TEXT)
        print("MODEL_GENERATION_READY:", MODEL_GENERATION_READY)
        print("SMOKE:", SMOKE_TEXT)
    except Exception:
        MODEL_ERROR = traceback.format_exc()
        MODEL_GENERATION_READY = False
        print("MODEL SMOKE TEST FAILED")
        print(MODEL_ERROR)

load_model()
model_smoke_test()

print("DEPENDENCY_STATUS:", DEPENDENCY_STATUS)
print("DEVICE_INFO:", DEVICE_INFO)


Loading model: Qwen/Qwen2.5-1.5B-Instruct


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


MODEL_READY: True DEVICE: cuda:0
MODEL_GENERATION_READY: True
SMOKE: READY
DEPENDENCY_STATUS: {'checked': True, 'attempted_install': False, 'missing_before': [], 'missing_after': [], 'errors': []}
DEVICE_INFO: {'torch_version': '2.11.0+cu126', 'cuda_available': True, 'device_count': 1, 'cuda_version': '12.6', 'gpu_name': 'NVIDIA GeForce RTX 4060'}


## 2. Runtime Objects

In [3]:

class State(Enum):
    PSI = "Ψ"
    OMEGA = "Ω"
    BOTTOM = "⊥"

@dataclass
class InputInductionPacket:
    raw_input: str
    inferred_task: str
    preserved_function: str
    implied_operations: List[str]
    missing_slots: List[str]
    semantic_locks: Dict[str, str]
    forbidden_drifts: List[str]
    model_facing_prompt: str
    collapse_rule: str
    confidence: float
    packet_origin: str = "model"

@dataclass
class RuntimeContract:
    profile: str
    raw_input: str
    inferred_task: str
    preserved_function: str
    preconditions: List[str]
    postconditions: List[str]
    success_criteria: List[str]
    failure_criteria: List[str]
    semantic_locks: Dict[str, str]
    forbidden_drifts: List[str]
    required_operations: List[str]
    rollback_mechanism: str
    trace_update: str
    mutation_history: List[Dict[str, Any]] = field(default_factory=list)

    def signature(self) -> str:
        content = json.dumps(asdict(self), sort_keys=True, ensure_ascii=False)
        return hashlib.sha256(content.encode("utf-8")).hexdigest()[:12]

@dataclass
class Branch:
    role: str
    origin: str
    depth: int
    prompt_used: str
    output: str
    score: float = 0.0
    dimensions: Dict[str, float] = field(default_factory=dict)
    checks: Dict[str, Any] = field(default_factory=dict)
    polysemy_check: bool = False
    rejected: bool = True
    rejection_reasons: List[str] = field(default_factory=list)

@dataclass
class ContractPatch:
    reason: str
    add_preconditions: List[str] = field(default_factory=list)
    add_postconditions: List[str] = field(default_factory=list)
    add_success_criteria: List[str] = field(default_factory=list)
    add_failure_criteria: List[str] = field(default_factory=list)
    add_required_operations: List[str] = field(default_factory=list)
    add_semantic_locks: Dict[str, str] = field(default_factory=dict)
    add_forbidden_drifts: List[str] = field(default_factory=list)
    rollback_update: str = ""
    trace_note: str = ""

@dataclass
class Residue:
    description: str
    missing_pieces: List[str]
    contract_patch: Dict[str, Any]
    induced_next_moves: List[str]
    residue_score: float
    actionable: bool
    dead_reason: Optional[str] = None

@dataclass
class DepthTrace:
    depth: int
    contract_signature: str
    state: str
    reason: str
    branch_scores: Dict[str, float]
    winner_role: Optional[str]
    residue: Optional[Dict[str, Any]]

@dataclass
class PromptResult:
    prompt: str
    state: str
    reason: str
    depth: int
    induction_packet: Dict[str, Any]
    final_contract: Dict[str, Any]
    contract_signatures: List[str]
    winner_branch: Optional[str]
    winner_origin: Optional[str]
    winner_score: float
    answer: str
    raw_answer: str
    shaped_answer: Optional[str]
    branches: List[Dict[str, Any]]
    residues: List[Dict[str, Any]]
    depth_trace: List[Dict[str, Any]]
    metrics: Dict[str, Any]


## 3. Utility Functions

In [4]:

def model_generate(user_prompt: str, max_new_tokens: int, temperature: float) -> str:
    if REQUIRE_MODEL_FOR_PSI and not MODEL_GENERATION_READY:
        raise RuntimeError("Model generation not ready; refusing fallback collapse.")
    messages = [{"role": "user", "content": user_prompt}]
    rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(rendered, return_tensors="pt").to(next(model.parameters()).device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True if temperature > 0 else False,
            return_dict_in_generate=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen = out.sequences[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

def normalize_text(s: Any) -> str:
    return re.sub(r"\s+", " ", str(s or "").lower()).strip()

def contains_any(text: str, terms: List[str]) -> bool:
    t = normalize_text(text)
    return any(term.lower() in t for term in terms)

def count_terms(text: str, terms: List[str]) -> int:
    t = normalize_text(text)
    return sum(1 for term in terms if term and term.lower() in t)

def word_count(text: str) -> int:
    return len(re.findall(r"\b\w+\b", str(text or "")))

def unique_append(base: List[str], new_items: List[str]) -> List[str]:
    seen = {x.strip().lower() for x in base if str(x).strip()}
    out = list(base)
    for item in new_items or []:
        s = str(item).strip()
        if s and s.lower() not in seen:
            out.append(s)
            seen.add(s.lower())
    return out

def extract_json_object(text: str) -> Optional[Dict[str, Any]]:
    if not text:
        return None

    # Direct parse first.
    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass

    # Remove markdown fences.
    cleaned = re.sub(r"```(?:json|JSON)?", "", text).replace("```", "").strip()

    try:
        obj = json.loads(cleaned)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass

    # Find largest plausible object.
    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start >= 0 and end > start:
        candidate = cleaned[start:end+1]
        try:
            obj = json.loads(candidate)
            if isinstance(obj, dict):
                return obj
        except Exception:
            return None

    return None

def safe_list(x: Any) -> List[str]:
    if x is None:
        return []
    if isinstance(x, list):
        return [str(i).strip() for i in x if str(i).strip()]
    if isinstance(x, str):
        if not x.strip():
            return []
        return [x.strip()]
    return [str(x).strip()]

def safe_dict(x: Any) -> Dict[str, str]:
    if isinstance(x, dict):
        return {str(k).strip(): str(v).strip() for k, v in x.items() if str(k).strip() and str(v).strip()}
    return {}

def clamp01(x: Any, default: float = 0.5) -> float:
    try:
        return max(0.0, min(1.0, float(x)))
    except Exception:
        return default


## 4. Input Induction Compiler

This stage does not answer. It compiles the raw user signal into the correct internal input geometry.


In [5]:

DEFAULT_SEMANTIC_LOCKS = {
    "contract": "runtime execution contract, not legal agreement",
    "tool": "external action/function/API channel with possible side effects",
    "controller": "agent governance loop that owns policy and next-action selection",
    "policy": "runtime decision rule, not organizational ownership",
    "evidence": "observation submitted to verifier/controller, not a command",
    "memory": "causal trace continuity, not a paragraph summary",
    "retrieval": "operational fit / missing-slot recovery, not noun overlap",
    "shape": "operational structure, not literal geometry unless explicitly requested",
    "rollback": "restore prior valid state while preserving evidence and trace",
    "induction": "structured coupling that changes model trajectory without direct payload transfer",
    "residue": "unresolved structure that can induce the next operation",
}

PROFILE_KEYWORDS = {
    "runtime_contract": ["contract", "precondition", "postcondition", "success", "failure", "side effect", "rollback", "api call"],
    "tool_safety": ["tool", "safe", "unsafe", "permission", "risk", "side effect", "execute", "call"],
    "evidence_control": ["tool output", "tool result", "evidence", "observation", "controller", "policy", "authority", "command"],
    "state_recovery": ["rollback", "restore", "recover", "undo", "trace", "failed", "corrupted", "previous state"],
    "memory_trace": ["memory", "remember", "trace continuity", "summary", "history", "context", "which-path"],
    "inverse_retrieval": ["retrieval", "retrieve", "search", "keyword", "noun", "label", "operation", "missing slot", "inverse"],
    "input_induction": ["ask", "prompt", "input", "induce", "induction", "model-facing", "user input"],
    "recursive_solver": ["recursive", "residue", "keep solving", "loop", "next move", "discovery", "branch"],
}

def infer_profile(raw_input: str) -> str:
    t = normalize_text(raw_input)
    scores = {p: count_terms(t, terms) for p, terms in PROFILE_KEYWORDS.items()}
    if "tool output" in t or "tool result" in t:
        scores["evidence_control"] += 3
    if "how we ask" in t or "correct input" in t or "user input" in t:
        scores["input_induction"] += 4
    if "keep solving" in t or "recurs" in t or "residue" in t:
        scores["recursive_solver"] += 4
    if "contract" in t and "tool" in t:
        scores["runtime_contract"] += 3
    if "retriev" in t or "noun" in t or "keyword" in t:
        scores["inverse_retrieval"] += 3
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "general"

def deterministic_packet(raw_input: str) -> InputInductionPacket:
    profile = infer_profile(raw_input)
    t = normalize_text(raw_input)
    implied_ops = []
    missing_slots = []
    forbidden = []
    locks = dict(DEFAULT_SEMANTIC_LOCKS)

    if profile == "input_induction":
        implied_ops = ["infer true task", "compile model-facing prompt", "extract semantic locks", "avoid answering raw wording directly"]
        missing_slots = ["explicit internal task", "preserved function", "forbidden drift list"]
        forbidden = ["answer raw words directly", "generic prompt rewrite", "surface paraphrase"]
        inferred = "compile raw user input into a model-facing induction packet"
        preserved = "convert messy human signal into correct internal task geometry before answering"
    elif profile == "recursive_solver":
        implied_ops = ["solve current residue", "mutate contract", "rerun branches", "stop at Ψ/Ω/⊥"]
        missing_slots = ["residue shape", "contract patch", "next operation"]
        forbidden = ["recurse on text", "keep talking without residue", "false Ψ"]
        inferred = "run recursive residue-solving instead of answer-looping"
        preserved = "continue only where unresolved residue has actionable structure"
    elif profile == "inverse_retrieval":
        implied_ops = ["extract operation", "infer missing slot", "rank by function", "reject noun-only matches"]
        missing_slots = ["operation carrier", "candidate verifier", "negative neighbor"]
        forbidden = ["keyword-only search", "noun overlap", "literal shape drift"]
        inferred = "retrieve by inverse operational fit"
        preserved = "find what closes the missing operation, not what matches the surface noun"
    elif profile == "memory_trace":
        implied_ops = ["preserve causal trace", "separate memory from summary", "track state transitions"]
        missing_slots = ["state history", "decision trace", "update rule"]
        forbidden = ["memory as paragraph summary", "erase which-path information"]
        inferred = "model memory as trace continuity"
        preserved = "retain causal event path across turns"
    elif profile == "tool_safety":
        implied_ops = ["check permission", "check preconditions", "bound side effects", "define rollback"]
        missing_slots = ["risk gate", "side-effect boundary", "safe failure path"]
        forbidden = ["execute first", "unbounded tool action", "tool output as authority"]
        inferred = "gate tool use before execution"
        preserved = "prevent unsafe tool action by contract"
    elif profile == "evidence_control":
        implied_ops = ["treat outputs as evidence", "verify observation", "controller selects next action"]
        missing_slots = ["evidence gate", "policy lock", "conflict handler"]
        forbidden = ["tool output commands agent", "administrator policy drift", "legal-contract drift"]
        inferred = "control tool-output interpretation"
        preserved = "tool results inform but do not command"
    else:
        implied_ops = ["identify task", "preserve operation", "avoid semantic drift"]
        missing_slots = ["explicit preserved function", "success criteria"]
        forbidden = ["generic answer", "wrong domain"]
        inferred = "answer with operation preserved"
        preserved = "solve the user request without drifting into adjacent meanings"

    model_prompt = (
        f"Task: {inferred}\n"
        f"Preserved function: {preserved}\n"
        f"Required operations: {', '.join(implied_ops)}\n"
        f"Missing slots: {', '.join(missing_slots)}\n"
        f"Forbidden drift: {', '.join(forbidden)}\n"
        f"Collapse rule: prefer Ω over false Ψ."
    )

    return InputInductionPacket(
        raw_input=raw_input,
        inferred_task=inferred,
        preserved_function=preserved,
        implied_operations=implied_ops,
        missing_slots=missing_slots,
        semantic_locks=locks,
        forbidden_drifts=forbidden,
        model_facing_prompt=model_prompt,
        collapse_rule="prefer Ω over false Ψ; recurse on residue, not answers",
        confidence=0.55,
        packet_origin="deterministic_patch",
    )

def validate_packet_object(obj: Dict[str, Any], raw_input: str) -> Tuple[InputInductionPacket, List[str]]:
    warnings = []
    fallback = deterministic_packet(raw_input)

    packet = InputInductionPacket(
        raw_input=raw_input,
        inferred_task=str(obj.get("inferred_task") or fallback.inferred_task).strip(),
        preserved_function=str(obj.get("preserved_function") or fallback.preserved_function).strip(),
        implied_operations=safe_list(obj.get("implied_operations")) or fallback.implied_operations,
        missing_slots=safe_list(obj.get("missing_slots")) or fallback.missing_slots,
        semantic_locks={**fallback.semantic_locks, **safe_dict(obj.get("semantic_locks"))},
        forbidden_drifts=safe_list(obj.get("forbidden_drifts")) or fallback.forbidden_drifts,
        model_facing_prompt=str(obj.get("model_facing_prompt") or fallback.model_facing_prompt).strip(),
        collapse_rule=str(obj.get("collapse_rule") or fallback.collapse_rule).strip(),
        confidence=clamp01(obj.get("confidence", fallback.confidence), fallback.confidence),
        packet_origin="model",
    )

    if len(packet.inferred_task) < 8:
        warnings.append("inferred_task_short")
        packet.inferred_task = fallback.inferred_task
    if len(packet.preserved_function) < 8:
        warnings.append("preserved_function_short")
        packet.preserved_function = fallback.preserved_function
    if not packet.implied_operations:
        warnings.append("missing_implied_operations")
        packet.implied_operations = fallback.implied_operations
    if not packet.model_facing_prompt:
        warnings.append("missing_model_facing_prompt")
        packet.model_facing_prompt = fallback.model_facing_prompt

    return packet, warnings

def compile_input_induction_packet(raw_input: str) -> Tuple[InputInductionPacket, Dict[str, Any]]:
    fallback = deterministic_packet(raw_input)

    prompt = f"""
You are the RHI Input Induction Compiler.

Do not answer the user.
Compile the raw user input into the internal question/contract geometry.

Return strict JSON only with these fields:
- inferred_task: string
- preserved_function: string
- implied_operations: list of strings
- missing_slots: list of strings
- semantic_locks: object mapping term to required meaning
- forbidden_drifts: list of strings
- model_facing_prompt: string
- collapse_rule: string
- confidence: number from 0 to 1

Raw user input:
{raw_input}

Important locks:
- User raw words are not automatically the true task.
- Compile the correct model-facing input.
- Preserve verbs/operations over nouns.
- Prefer Ω over false Ψ.
- Recursion should happen on residue, not on answer text.
""".strip()

    meta = {
        "model_attempted": False,
        "model_raw": None,
        "json_parse_ok": False,
        "warnings": [],
        "fallback_used": False,
    }

    if MODEL_GENERATION_READY:
        try:
            meta["model_attempted"] = True
            raw = model_generate(prompt, MAX_NEW_TOKENS_INDUCTION, TEMPERATURE_INDUCTION)
            meta["model_raw"] = raw
            obj = extract_json_object(raw)
            if obj is not None:
                packet, warnings = validate_packet_object(obj, raw_input)
                packet.packet_origin = "model"
                meta["json_parse_ok"] = True
                meta["warnings"] = warnings
                if warnings and packet.confidence < 0.6:
                    # Keep model packet but mark lower confidence.
                    pass
                return packet, meta
            else:
                meta["warnings"].append("json_parse_failed")
        except Exception:
            meta["warnings"].append("model_induction_exception")
            meta["error"] = traceback.format_exc()

    meta["fallback_used"] = True
    return fallback, meta


## 5. Contract Builder and Contract Mutation

In [6]:

def build_contract(packet: InputInductionPacket, profile: Optional[str] = None) -> RuntimeContract:
    profile = profile or infer_profile(packet.raw_input + " " + packet.inferred_task + " " + packet.model_facing_prompt)

    preconditions = [
        "Input induction packet exists",
        "Preserved function is explicit",
        "Semantic locks are available before branch generation",
    ]
    postconditions = [
        "Answer preserves the induced task operation",
        "No forbidden semantic drift is introduced",
        "Residue is either resolved, shaped, or isolated",
    ]
    success = [
        "Branch answer addresses the model-facing prompt",
        "Branch answer satisfies required operations",
        "Operational critic accepts model-origin answer",
    ]
    failure = [
        "Generic answer",
        "Wrong semantic carrier",
        "Contract/spec echo without operational content",
        "Non-model origin candidate attempts collapse",
    ]

    if profile == "input_induction":
        preconditions.append("Raw user input must be compiled before answer generation")
        success.append("Compiled model-facing prompt is used instead of raw wording")
        failure.append("Assistant answers raw wording without induction")
    elif profile == "recursive_solver":
        preconditions.append("Residue must be identified before recursion")
        success.append("Contract is mutated by residue patch")
        failure.append("Recursion appends repair text without changing contract")
    elif profile == "tool_safety":
        preconditions.append("Tool permissions and side effects are checked before action")
        success.append("Unsafe tool calls are rejected or safely failed")
        failure.append("Tool executes before contract gate")
    elif profile == "evidence_control":
        preconditions.append("Tool output is treated as evidence, not command")
        success.append("Controller interprets observation before next action")
        failure.append("Tool result becomes authority")
    elif profile == "memory_trace":
        success.append("Memory is represented as causal trace continuity")
        failure.append("Memory is reduced to summary")
    elif profile == "inverse_retrieval":
        success.append("Candidate is selected by operational fit")
        failure.append("Keyword/noun overlap overrides preserved function")

    return RuntimeContract(
        profile=profile,
        raw_input=packet.raw_input,
        inferred_task=packet.inferred_task,
        preserved_function=packet.preserved_function,
        preconditions=preconditions,
        postconditions=postconditions,
        success_criteria=success,
        failure_criteria=failure,
        semantic_locks=dict(packet.semantic_locks),
        forbidden_drifts=list(packet.forbidden_drifts),
        required_operations=list(packet.implied_operations),
        rollback_mechanism="Return Ω with shaped residue and contract patch; never collapse fallback output as Ψ.",
        trace_update="Record induction packet, contract signature, branch audits, residue patch, and collapse reason.",
        mutation_history=[],
    )

def apply_patch(contract: RuntimeContract, patch: ContractPatch, depth: int) -> RuntimeContract:
    updated = RuntimeContract(**asdict(contract))
    updated.preconditions = unique_append(updated.preconditions, patch.add_preconditions)
    updated.postconditions = unique_append(updated.postconditions, patch.add_postconditions)
    updated.success_criteria = unique_append(updated.success_criteria, patch.add_success_criteria)
    updated.failure_criteria = unique_append(updated.failure_criteria, patch.add_failure_criteria)
    updated.required_operations = unique_append(updated.required_operations, patch.add_required_operations)
    updated.forbidden_drifts = unique_append(updated.forbidden_drifts, patch.add_forbidden_drifts)

    updated.semantic_locks = dict(updated.semantic_locks)
    for k, v in patch.add_semantic_locks.items():
        updated.semantic_locks[str(k)] = str(v)

    if patch.rollback_update:
        updated.rollback_mechanism = patch.rollback_update
    if patch.trace_note:
        updated.trace_update = updated.trace_update + " | " + patch.trace_note

    updated.mutation_history = list(updated.mutation_history)
    updated.mutation_history.append({
        "depth": depth,
        "patch": asdict(patch),
        "new_signature": updated.signature(),
    })
    return updated


## 6. Model-Origin Branch Engine

In [7]:

def contract_prompt_block(contract: RuntimeContract) -> str:
    locks = "\n".join([f"- {k}: {v}" for k, v in contract.semantic_locks.items()])
    return f"""
PROFILE: {contract.profile}
INFERRED TASK: {contract.inferred_task}
PRESERVED FUNCTION: {contract.preserved_function}

REQUIRED OPERATIONS:
{chr(10).join("- " + x for x in contract.required_operations)}

PRECONDITIONS:
{chr(10).join("- " + x for x in contract.preconditions)}

POSTCONDITIONS:
{chr(10).join("- " + x for x in contract.postconditions)}

SUCCESS CRITERIA:
{chr(10).join("- " + x for x in contract.success_criteria)}

FAILURE CRITERIA:
{chr(10).join("- " + x for x in contract.failure_criteria)}

SEMANTIC LOCKS:
{locks}

FORBIDDEN DRIFTS:
{chr(10).join("- " + x for x in contract.forbidden_drifts)}

ROLLBACK:
{contract.rollback_mechanism}
""".strip()

def branch_instruction(role: str) -> str:
    return {
        "construct": "Build the direct operational answer that satisfies the contract.",
        "verify": "Verify the induced task, reject wrong carriers, then answer.",
        "counter": "Challenge likely false assumptions and expose drift before answering.",
        "repair": "Repair missing contract dimensions before answering.",
        "explore": "Probe adjacent unknowns only if they create actionable next moves, then answer.",
    }.get(role, "Answer while preserving the contract.")

def make_branch_prompt(contract: RuntimeContract, role: str, residue_context: Optional[Residue] = None) -> str:
    residue_block = ""
    if residue_context is not None:
        residue_block = f"""
CURRENT RESIDUE:
{json.dumps(asdict(residue_context), indent=2, ensure_ascii=False)}

Use the residue to respect the current contract patch. Do not merely restate the residue.
""".strip()

    return f"""
You are one branch inside the RHI recursive solver.

RAW USER INPUT:
{contract.raw_input}

MODEL-FACING TASK:
{contract.inferred_task}

CONTRACT:
{contract_prompt_block(contract)}

BRANCH ROLE:
{role} — {branch_instruction(role)}

{residue_block}

OUTPUT RULES:
- Answer the model-facing task, not just the raw wording.
- Preserve the preserved function.
- Use semantic locks.
- Do not recite the contract unless the user explicitly requested a schema.
- Do not use legal-contract language unless legally asked.
- Prefer Ω-style uncertainty over false Ψ if the task cannot be preserved.
""".strip()

def generate_branch(contract: RuntimeContract, role: str, depth: int, residue_context: Optional[Residue]) -> Branch:
    prompt = make_branch_prompt(contract, role, residue_context)
    try:
        output = model_generate(prompt, MAX_NEW_TOKENS_BRANCH, TEMPERATURE_BRANCH)
        return Branch(
            role=role,
            origin="model",
            depth=depth,
            prompt_used=prompt,
            output=output,
        )
    except Exception:
        return Branch(
            role=role,
            origin="error",
            depth=depth,
            prompt_used=prompt,
            output="",
            score=0.0,
            dimensions={},
            checks={"error": traceback.format_exc()},
            polysemy_check=False,
            rejected=True,
            rejection_reasons=["model_generation_error"],
        )


## 7. Operational Critic

The critic does not grade absolute truth. It grades operational fit against the induced contract.


In [8]:

LEGAL_DRIFT_TERMS = [
    "legal agreement", "contract law", "liability", "stakeholder", "terms of service",
    "party to the contract", "parties", "hereby", "binding agreement", "signatory"
]
ADMIN_POLICY_DRIFT = ["system administrator", "security team", "company policy", "organizational policy"]
TOOL_COMMAND_DRIFT = ["tool decides", "tool should decide", "output commands", "tool output controls", "tool result controls"]
SUMMARY_ONLY_DRIFT = ["memory is just a summary", "conversation summary is memory", "simple recap is memory"]
KEYWORD_ONLY_DRIFT = ["keyword-only", "noun overlap only", "title match only"]
FALLBACK_DRIFT = ["diagnostic fallback", "fallback output", "not a model answer"]

PROFILE_THRESHOLDS = {
    "input_induction": 0.70,
    "recursive_solver": 0.70,
    "runtime_contract": 0.70,
    "tool_safety": 0.70,
    "evidence_control": 0.70,
    "state_recovery": 0.69,
    "memory_trace": 0.68,
    "inverse_retrieval": 0.68,
    "general": 0.70,
}

def audit_branch(branch: Branch, contract: RuntimeContract) -> Branch:
    text = normalize_text(branch.output)
    wc = word_count(branch.output)

    required_hits = count_terms(text, contract.required_operations)
    required_ratio = required_hits / max(1, len(contract.required_operations))

    success_hits = count_terms(text, contract.success_criteria)
    success_ratio = success_hits / max(1, len(contract.success_criteria))

    lock_hits = 0
    for k in contract.semantic_locks:
        if k.lower() in text:
            lock_hits += 1
    lock_ratio = lock_hits / max(1, min(8, len(contract.semantic_locks)))

    forbidden_hits = []
    for term in contract.forbidden_drifts:
        if term and term.lower() in text:
            forbidden_hits.append(term)

    legal_drift = contains_any(text, LEGAL_DRIFT_TERMS)
    admin_policy_drift = contract.profile == "evidence_control" and contains_any(text, ADMIN_POLICY_DRIFT)
    tool_command_drift = contract.profile == "evidence_control" and contains_any(text, TOOL_COMMAND_DRIFT)
    summary_only_drift = contract.profile == "memory_trace" and contains_any(text, SUMMARY_ONLY_DRIFT)
    keyword_only_drift = contract.profile == "inverse_retrieval" and contains_any(text, KEYWORD_ONLY_DRIFT)
    fallback_drift = contains_any(text, FALLBACK_DRIFT)

    profile_checks = {}

    if contract.profile == "input_induction":
        profile_checks = {
            "compiled_not_raw": contains_any(text, ["model-facing", "internal task", "induce", "compiled", "contract", "slot"]),
            "preserved_function": contains_any(text, ["preserved function", "operation", "intent", "task"]),
            "forbidden_drift": contains_any(text, ["drift", "forbidden", "wrong", "avoid"]),
            "not_prompt_tips_only": not contains_any(text, ["just rephrase", "prompt engineering tips"]),
        }
    elif contract.profile == "recursive_solver":
        profile_checks = {
            "residue": contains_any(text, ["residue", "unresolved", "missing", "gap"]),
            "contract_mutation": contains_any(text, ["contract", "patch", "mutate", "update", "constraint"]),
            "not_answer_loop": contains_any(text, ["not answer", "not just", "recurse on residue", "next move"]),
            "stop_condition": contains_any(text, ["ψ", "omega", "Ω", "bottom", "⊥", "collapse"]),
        }
    elif contract.profile == "runtime_contract":
        profile_checks = {
            "runtime_not_legal": not legal_drift and contains_any(text, ["runtime", "execution", "tool", "function", "api"]),
            "preconditions": contains_any(text, ["precondition", "before", "prerequisite"]),
            "postconditions": contains_any(text, ["postcondition", "after", "verify", "expected state"]),
            "side_effects": contains_any(text, ["side effect", "bounded", "scope"]),
            "rollback": contains_any(text, ["rollback", "restore", "recover", "safe failure"]),
        }
    elif contract.profile == "tool_safety":
        profile_checks = {
            "permission": contains_any(text, ["permission", "authorized", "allowed"]),
            "precondition": contains_any(text, ["precondition", "before", "prerequisite"]),
            "risk": contains_any(text, ["risk", "unsafe", "danger", "impact"]),
            "side_effect": contains_any(text, ["side effect", "bounded", "scope"]),
            "safe_failure": contains_any(text, ["reject", "abort", "rollback", "safe failure"]),
        }
    elif contract.profile == "evidence_control":
        profile_checks = {
            "evidence_not_command": contains_any(text, ["evidence", "observation", "inform"]) and not tool_command_drift,
            "controller": contains_any(text, ["controller", "agent", "runtime"]) and not admin_policy_drift,
            "policy_runtime": contains_any(text, ["policy", "decision rule", "gate", "contract"]) and not admin_policy_drift,
            "verify": contains_any(text, ["verify", "validate", "check", "corroborate"]),
            "next_action_owned": contains_any(text, ["next action", "decide", "interpret", "selection"]),
        }
    elif contract.profile == "memory_trace":
        profile_checks = {
            "not_summary": contains_any(text, ["not a summary", "more than a summary", "not just", "summary loses"]),
            "causal_trace": contains_any(text, ["trace", "causal", "which-path", "state transition"]),
            "observations_decisions": contains_any(text, ["observation", "decision", "action", "result"]),
            "state_update": contains_any(text, ["update", "continuity", "state", "history"]),
        }
    elif contract.profile == "inverse_retrieval":
        profile_checks = {
            "operation": contains_any(text, ["operation", "function", "action", "transformation", "affordance"]),
            "missing_slot": contains_any(text, ["missing", "slot", "need", "inverse", "desired effect"]),
            "candidate": contains_any(text, ["candidate", "retrieve", "search", "rank"]),
            "reject_noun": contains_any(text, ["reject", "not keyword", "not noun", "surface", "label"]),
            "verify_fit": contains_any(text, ["verify", "preserve", "fit", "closes", "works"]),
        }
    else:
        profile_checks = {
            "substantive": wc >= 40,
            "specific": required_ratio > 0.25 or success_ratio > 0.20,
            "no_bad_drift": not legal_drift and not fallback_drift,
        }

    profile_quality = sum(bool(v) for v in profile_checks.values()) / max(1, len(profile_checks))

    length_score = min(1.0, wc / 100)
    if wc > 300:
        length_score -= min(0.25, (wc - 300) / 800)

    drift_flags = {
        "legal_drift": legal_drift,
        "admin_policy_drift": admin_policy_drift,
        "tool_command_drift": tool_command_drift,
        "summary_only_drift": summary_only_drift,
        "keyword_only_drift": keyword_only_drift,
        "fallback_drift": fallback_drift,
    }

    drift_penalty = 0.0
    drift_penalty += 0.18 * sum(1 for v in drift_flags.values() if v)
    drift_penalty += 0.06 * len(forbidden_hits)

    origin_score = 1.0 if branch.origin == "model" else 0.0

    score = (
        0.35 * profile_quality
        + 0.18 * required_ratio
        + 0.12 * success_ratio
        + 0.10 * lock_ratio
        + 0.12 * length_score
        + 0.13 * origin_score
        - drift_penalty
    )
    score = max(0.0, min(1.0, float(score)))

    branch.score = score
    branch.dimensions = {
        "profile_quality": profile_quality,
        "required_ratio": required_ratio,
        "success_ratio": success_ratio,
        "lock_ratio": lock_ratio,
        "length_score": length_score,
        "origin_score": origin_score,
        "drift_penalty": drift_penalty,
    }
    branch.checks = {
        "profile_checks": profile_checks,
        "forbidden_hits": forbidden_hits,
        "drift_flags": drift_flags,
        "word_count": wc,
    }

    branch.polysemy_check = not any([
        legal_drift,
        admin_policy_drift,
        tool_command_drift,
        summary_only_drift,
        keyword_only_drift,
        fallback_drift,
    ])

    threshold = PROFILE_THRESHOLDS.get(contract.profile, 0.70)
    reasons = []
    if branch.origin != "model":
        reasons.append("origin_not_model")
    if wc < 25:
        reasons.append("too_short")
    if not branch.polysemy_check:
        reasons.append("polysemy_or_origin_drift")
    if forbidden_hits:
        reasons.append("forbidden_hits:" + ",".join(forbidden_hits))
    if profile_quality < 0.45:
        reasons.append("profile_quality_low")
    if score < threshold:
        reasons.append("below_threshold")

    branch.rejected = bool(reasons)
    branch.rejection_reasons = reasons
    return branch


## 8. Residue Engine and Contract Patch Compiler

In [9]:

def fallback_patch_from_residue(contract: RuntimeContract, branches: List[Branch], depth: int) -> ContractPatch:
    all_reasons = []
    failed_checks = Counter()
    drift_terms = Counter()

    for b in branches:
        all_reasons.extend(b.rejection_reasons)
        for k, v in b.checks.get("profile_checks", {}).items():
            if not v:
                failed_checks[k] += 1
        for k, v in b.checks.get("drift_flags", {}).items():
            if v:
                drift_terms[k] += 1

    patch = ContractPatch(
        reason="deterministic_patch_from_residue",
        trace_note=f"depth {depth}: patched from failed checks {dict(failed_checks)} and drift {dict(drift_terms)}",
    )

    if "origin_not_model" in all_reasons:
        patch.add_failure_criteria.append("Collapse requires model-origin branch output")
        patch.add_required_operations.append("verify branch origin before collapse")

    if drift_terms.get("legal_drift", 0) > 0:
        patch.add_semantic_locks["contract"] = "runtime execution contract only; exclude legal agreement language"
        patch.add_forbidden_drifts.extend(["legal agreement", "binding parties", "liability", "contract law"])

    if drift_terms.get("admin_policy_drift", 0) > 0:
        patch.add_semantic_locks["policy"] = "runtime decision rule owned by agent controller"
        patch.add_forbidden_drifts.extend(["administrator policy", "security team owns policy", "company policy"])

    if drift_terms.get("tool_command_drift", 0) > 0:
        patch.add_semantic_locks["evidence"] = "tool output is observation/evidence, never command authority"
        patch.add_required_operations.append("route evidence through controller before next action")
        patch.add_forbidden_drifts.extend(["tool output controls next action", "tool decides"])

    if drift_terms.get("summary_only_drift", 0) > 0:
        patch.add_semantic_locks["memory"] = "causal trace continuity, not text summary"
        patch.add_required_operations.append("preserve state transitions and which-path information")
        patch.add_forbidden_drifts.append("memory is just a summary")

    if drift_terms.get("keyword_only_drift", 0) > 0:
        patch.add_semantic_locks["retrieval"] = "operational fit by missing slot, not noun overlap"
        patch.add_required_operations.append("verify candidate by function preserved")
        patch.add_forbidden_drifts.append("keyword-only retrieval")

    for check, count in failed_checks.most_common(4):
        patch.add_success_criteria.append(f"repair failed check: {check}")
        patch.add_required_operations.append(f"satisfy {check}")

    if not patch.add_required_operations:
        patch.add_required_operations.append("answer with more explicit operational fit")
    if not patch.add_success_criteria:
        patch.add_success_criteria.append("raise profile quality above collapse threshold")

    return patch

def compute_residue(contract: RuntimeContract, branches: List[Branch], depth: int) -> Residue:
    best = max(branches, key=lambda b: b.score) if branches else None
    threshold = PROFILE_THRESHOLDS.get(contract.profile, 0.70)
    best_score = best.score if best else 0.0

    missing = []
    next_moves = []

    if not best or best.origin != "model":
        missing.append("no model-origin winner")
        next_moves.append("regenerate model-origin branches")
    if best and best.score < threshold:
        missing.append(f"best score {best.score:.3f} below threshold {threshold:.3f}")
        next_moves.append("mutate contract to address failed audit dimensions")
    if best and not best.polysemy_check:
        missing.append("polysemy or origin drift")
        next_moves.append("add explicit semantic lock and forbidden drift terms")

    failed_checks = Counter()
    for b in branches:
        for k, v in b.checks.get("profile_checks", {}).items():
            if not v:
                failed_checks[k] += 1
    for k, c in failed_checks.most_common(3):
        missing.append(f"profile check failed: {k}")
        next_moves.append(f"patch contract to require {k}")

    patch = fallback_patch_from_residue(contract, branches, depth)

    residue_score = min(1.0, max(0.0, 1.0 - best_score + 0.05 * depth))
    actionable = bool(next_moves) and residue_score < 0.95 and depth < MAX_DEPTH

    return Residue(
        description=f"depth {depth}: unresolved residue after branch audit",
        missing_pieces=unique_append([], missing),
        contract_patch=asdict(patch),
        induced_next_moves=unique_append([], next_moves),
        residue_score=residue_score,
        actionable=actionable,
        dead_reason=None if actionable else "no_actionable_patch_or_max_depth",
    )

def compile_model_patch(contract: RuntimeContract, residue: Residue, depth: int) -> Tuple[ContractPatch, Dict[str, Any]]:
    fallback = ContractPatch(**residue.contract_patch)
    meta = {
        "model_attempted": False,
        "json_parse_ok": False,
        "model_raw": None,
        "fallback_used": False,
        "warnings": [],
    }

    prompt = f"""
You are the RHI Contract Patch Compiler.

Do not answer the user.
Given the current contract and residue, output a JSON contract patch.

Patch JSON fields:
- reason: string
- add_preconditions: list of strings
- add_postconditions: list of strings
- add_success_criteria: list of strings
- add_failure_criteria: list of strings
- add_required_operations: list of strings
- add_semantic_locks: object mapping term to required meaning
- add_forbidden_drifts: list of strings
- rollback_update: string
- trace_note: string

CURRENT CONTRACT:
{json.dumps(asdict(contract), indent=2, ensure_ascii=False)}

RESIDUE:
{json.dumps(asdict(residue), indent=2, ensure_ascii=False)}

Rules:
- Mutate the contract, not the raw prompt text.
- Patch the specific failed checks.
- Add semantic locks for polysemy drift.
- Prefer Ω over false Ψ.
- Return strict JSON only.
""".strip()

    if MODEL_GENERATION_READY:
        try:
            meta["model_attempted"] = True
            raw = model_generate(prompt, MAX_NEW_TOKENS_PATCH, TEMPERATURE_PATCH)
            meta["model_raw"] = raw
            obj = extract_json_object(raw)
            if obj:
                patch = ContractPatch(
                    reason=str(obj.get("reason") or fallback.reason),
                    add_preconditions=safe_list(obj.get("add_preconditions")) or fallback.add_preconditions,
                    add_postconditions=safe_list(obj.get("add_postconditions")) or fallback.add_postconditions,
                    add_success_criteria=safe_list(obj.get("add_success_criteria")) or fallback.add_success_criteria,
                    add_failure_criteria=safe_list(obj.get("add_failure_criteria")) or fallback.add_failure_criteria,
                    add_required_operations=safe_list(obj.get("add_required_operations")) or fallback.add_required_operations,
                    add_semantic_locks=safe_dict(obj.get("add_semantic_locks")) or fallback.add_semantic_locks,
                    add_forbidden_drifts=safe_list(obj.get("add_forbidden_drifts")) or fallback.add_forbidden_drifts,
                    rollback_update=str(obj.get("rollback_update") or fallback.rollback_update),
                    trace_note=str(obj.get("trace_note") or fallback.trace_note),
                )
                meta["json_parse_ok"] = True
                return patch, meta
            meta["warnings"].append("json_parse_failed")
        except Exception:
            meta["warnings"].append("model_patch_exception")
            meta["error"] = traceback.format_exc()

    meta["fallback_used"] = True
    return fallback, meta


## 9. Collapse Gate and Payload Shaper

In [10]:

def collapse_gate(contract: RuntimeContract, branches: List[Branch]) -> Tuple[State, str, Optional[Branch], Dict[str, Any]]:
    if not branches:
        return State.OMEGA, "no_branches", None, {}

    ordered = sorted(branches, key=lambda b: b.score, reverse=True)
    best = ordered[0]
    second = ordered[1] if len(ordered) > 1 else None
    threshold = PROFILE_THRESHOLDS.get(contract.profile, 0.70)
    margin = best.score - (second.score if second else 0.0)

    valid = [b for b in ordered if b.origin == "model" and not b.rejected and b.polysemy_check]
    valid_roles = [b.role for b in valid]

    meta = {
        "threshold": threshold,
        "best_score": best.score,
        "margin": margin,
        "valid_count": len(valid),
        "valid_roles": valid_roles,
    }

    if best.origin != "model":
        return State.OMEGA, "winner_not_model_origin", best, meta

    if best.rejected or not best.polysemy_check:
        return State.OMEGA, "winner_rejected_or_polysemy_failed", best, meta

    if best.score >= threshold and margin >= 0.045:
        return State.PSI, "direct_margin_collapse", best, meta

    if len(valid) >= 2:
        top_valid = valid[:3]
        mean_score = float(np.mean([b.score for b in top_valid]))
        # Operational agreement: shared passed checks, not surface text.
        all_keys = set()
        for b in top_valid:
            all_keys |= set(b.checks.get("profile_checks", {}).keys())
        agree = 0
        for k in all_keys:
            if sum(bool(b.checks.get("profile_checks", {}).get(k, False)) for b in top_valid) >= 2:
                agree += 1
        op_agreement = agree / max(1, len(all_keys))
        meta["consensus_mean_score"] = mean_score
        meta["op_agreement"] = op_agreement

        if mean_score >= threshold - 0.03 and op_agreement >= 0.60:
            return State.PSI, "operational_consensus_collapse", top_valid[0], meta

    return State.OMEGA, "collapse_threshold_not_met", best, meta

def shape_payload(contract: RuntimeContract, raw_answer: str, raw_score: float) -> Tuple[str, bool, Dict[str, Any]]:
    if not SHAPER_ENABLED:
        return raw_answer, False, {"reason": "shaper_disabled"}

    prompt = f"""
Compress the accepted answer without changing its operational meaning.

RAW USER INPUT:
{contract.raw_input}

PRESERVED FUNCTION:
{contract.preserved_function}

SEMANTIC LOCKS:
{json.dumps(contract.semantic_locks, indent=2, ensure_ascii=False)}

RAW ACCEPTED ANSWER:
{raw_answer}

Rules:
- Preserve the operation and semantic locks.
- Do not introduce forbidden drift.
- Do not turn runtime terms into legal/organizational meanings.
- Aim for 50-130 words unless the user asked for a schema.
""".strip()

    try:
        shaped = model_generate(prompt, MAX_NEW_TOKENS_SHAPER, TEMPERATURE_SHAPER)
        tmp = Branch(
            role="shaper",
            origin="model",
            depth=999,
            prompt_used=prompt,
            output=shaped,
        )
        tmp = audit_branch(tmp, contract)
        raw_wc = word_count(raw_answer)
        shaped_wc = word_count(shaped)
        compression = 1.0 - shaped_wc / max(1, raw_wc)
        accepted = (
            tmp.origin == "model"
            and tmp.polysemy_check
            and tmp.score >= max(PROFILE_THRESHOLDS.get(contract.profile, 0.70) - 0.05, raw_score - 0.12)
            and shaped_wc <= max(raw_wc + 10, 145)
        )
        return (shaped if accepted else raw_answer), bool(accepted), {
            "reason": "accepted" if accepted else "rejected",
            "raw_words": raw_wc,
            "shaped_words": shaped_wc,
            "compression_ratio": compression,
            "audit": asdict(tmp),
        }
    except Exception:
        return raw_answer, False, {"reason": "shaper_error", "error": traceback.format_exc()}


## 10. Recursive Solver

In [11]:

def solve_prompt(raw_input: str) -> PromptResult:
    packet, packet_meta = compile_input_induction_packet(raw_input)
    profile = infer_profile(raw_input + " " + packet.inferred_task + " " + packet.model_facing_prompt)
    contract = build_contract(packet, profile)

    all_branches: List[Branch] = []
    residues: List[Residue] = []
    depth_trace: List[DepthTrace] = []
    contract_signatures = [contract.signature()]
    current_residue: Optional[Residue] = None

    final_state = State.OMEGA
    final_reason = "not_started"
    winner: Optional[Branch] = None
    collapse_meta = {}

    for depth in range(MAX_DEPTH + 1):
        depth_branches = []
        for role in BRANCH_ROLES:
            b = generate_branch(contract, role, depth, current_residue)
            b = audit_branch(b, contract)
            depth_branches.append(b)
            all_branches.append(b)

        state, reason, winner, collapse_meta = collapse_gate(contract, depth_branches)
        final_state, final_reason = state, reason

        if state == State.PSI:
            depth_trace.append(DepthTrace(
                depth=depth,
                contract_signature=contract.signature(),
                state=state.value,
                reason=reason,
                branch_scores={b.role: b.score for b in depth_branches},
                winner_role=winner.role if winner else None,
                residue=None,
            ))
            break

        residue = compute_residue(contract, depth_branches, depth)
        residues.append(residue)
        depth_trace.append(DepthTrace(
            depth=depth,
            contract_signature=contract.signature(),
            state=state.value,
            reason=reason,
            branch_scores={b.role: b.score for b in depth_branches},
            winner_role=winner.role if winner else None,
            residue=asdict(residue),
        ))

        if depth >= MAX_DEPTH or not residue.actionable:
            if not residue.actionable:
                final_state = State.BOTTOM
                final_reason = residue.dead_reason or "no_actionable_residue"
            break

        patch, patch_meta = compile_model_patch(contract, residue, depth)
        # Attach patch meta into residue for trace.
        residue.contract_patch["patch_meta"] = patch_meta
        contract = apply_patch(contract, patch, depth=depth)
        contract_signatures.append(contract.signature())
        current_residue = residue

    raw_answer = winner.output if winner else ""
    final_answer = raw_answer
    shaped_answer = None
    shaping_accepted = False
    shaper_meta = {}

    if final_state == State.PSI and winner is not None:
        final_answer, shaping_accepted, shaper_meta = shape_payload(contract, raw_answer, winner.score)
        shaped_answer = final_answer if shaping_accepted else None

    if REQUIRE_MODEL_FOR_PSI and final_state == State.PSI:
        if winner is None or winner.origin != "model":
            final_state = State.OMEGA
            final_reason = "model_origin_required_for_psi"
            final_answer = ""

    metrics = {
        "profile": contract.profile,
        "packet_meta": packet_meta,
        "contract_count": len(contract_signatures),
        "branch_count": len(all_branches),
        "rejected_branch_count": sum(1 for b in all_branches if b.rejected),
        "model_branch_count": sum(1 for b in all_branches if b.origin == "model"),
        "exhaust_ratio": sum(1 for b in all_branches if b.rejected) / max(1, len(all_branches)),
        "best_score": max([b.score for b in all_branches], default=0.0),
        "mean_score": float(np.mean([b.score for b in all_branches])) if all_branches else 0.0,
        "collapse_meta": collapse_meta,
        "shaping_accepted": shaping_accepted,
        "shaper_meta": shaper_meta,
    }

    return PromptResult(
        prompt=raw_input,
        state=final_state.value,
        reason=final_reason,
        depth=len(depth_trace) - 1,
        induction_packet=asdict(packet),
        final_contract=asdict(contract),
        contract_signatures=contract_signatures,
        winner_branch=winner.role if winner else None,
        winner_origin=winner.origin if winner else None,
        winner_score=winner.score if winner else 0.0,
        answer=final_answer,
        raw_answer=raw_answer,
        shaped_answer=shaped_answer,
        branches=[asdict(b) for b in all_branches],
        residues=[asdict(r) for r in residues],
        depth_trace=[asdict(d) for d in depth_trace],
        metrics=metrics,
    )


## 11. Prompt Battery

This battery is focused on the new AI branch: input induction, slot contracts, recursive residue-solving, memory, tool safety, and evidence control.


In [12]:

PROMPT_BATTERY = [
    # Input induction
    "another thing is how we ask the AI. we may need a AI to generate the correct input from the user input.",
    "convert messy user input into the correct model-facing prompt before answering.",
    "design an input compiler that turns implied user intent into a runtime contract.",
    "explain why the raw user prompt is not always the true task.",
    "build a prompt coil compiler that induces the right internal question.",
    "how should an AI ask itself the right question from a vague user request.",

    # Recursive residue solving
    "if its recursive it should just keep solving, not keep talking.",
    "design a recursive AI loop that recurses on residue not answers.",
    "explain how Ω residue becomes the next better question.",
    "build a residue engine that mutates the contract instead of appending repair text.",
    "when should a recursive solver stop and return bottom.",
    "explain discovery as shaped residue becoming the next operation.",

    # Runtime contract
    "explain why current AI agents fail when they use tools before forming a contract.",
    "design a runtime contract for a file-writing tool.",
    "explain success and failure criteria for an API call.",
    "build a tool-use contract for deleting a file.",
    "show how preconditions and postconditions bound a function call.",
    "explain why tool calls need rollback plans.",

    # Tool safety
    "how should an agent decide whether a tool call is safe.",
    "design a safety gate for an external API call.",
    "when should an agent reject a tool call.",
    "describe safe failure for a dangerous tool action.",
    "explain permission checks before tool execution.",
    "describe bounded risk for tool use in an agent.",

    # Evidence control
    "why is tool output evidence rather than the driver of the agent.",
    "describe tool output as observation not command.",
    "why should the controller own policy after a tool returns.",
    "design a verifier that treats API output as evidence.",
    "describe the difference between evidence and authority in tool use.",
    "how should an agent treat conflicting tool outputs.",

    # Memory trace
    "explain memory in an agent as trace continuity rather than a text summary.",
    "why is a conversation summary not the same as agent memory.",
    "describe memory as causal event history across turns.",
    "why does context amnesia break recursive agents.",
    "explain why summaries lose which-path information.",
    "how should recursive memory preserve state transitions observations decisions and updates.",

    # Inverse retrieval
    "design a shape-first retrieval step where no noun match exists but the inverse need is clear.",
    "how should retrieval work when keywords fail but the operation is obvious.",
    "explain inverse operational fit for search without noun matching.",
    "design a verifier for retrieval candidates selected by need rather than label.",
    "how can an agent rank candidates by function instead of name.",
    "build a retrieval step that rejects keyword-only matches.",
]

PROMPTS = PROMPT_BATTERY[:RUN_PROMPT_LIMIT]
print("Prompt count:", len(PROMPTS))


Prompt count: 36


## 12. Execute Run

In [13]:

if REQUIRE_MODEL_FOR_PSI and not MODEL_GENERATION_READY:
    raise RuntimeError("Model generation is not ready. v27 refuses fallback Ψ.")

t0 = time.time()
RESULTS: List[PromptResult] = []

for i, prompt in enumerate(PROMPTS, start=1):
    print("=" * 100)
    print(f"[{i}/{len(PROMPTS)}] {prompt}")
    try:
        result = solve_prompt(prompt)
    except Exception:
        err = traceback.format_exc()
        dummy_packet = deterministic_packet(prompt)
        dummy_contract = build_contract(dummy_packet)
        result = PromptResult(
            prompt=prompt,
            state=State.OMEGA.value,
            reason="kernel_exception",
            depth=0,
            induction_packet=asdict(dummy_packet),
            final_contract=asdict(dummy_contract),
            contract_signatures=[dummy_contract.signature()],
            winner_branch=None,
            winner_origin=None,
            winner_score=0.0,
            answer="",
            raw_answer="",
            shaped_answer=None,
            branches=[],
            residues=[],
            depth_trace=[],
            metrics={"error": err},
        )

    RESULTS.append(result)
    print("STATE:", result.state, "REASON:", result.reason, "DEPTH:", result.depth, "WINNER:", result.winner_branch, result.winner_score)
    print("PROFILE:", result.final_contract.get("profile"))
    print("ANSWER:", (result.answer or "")[:260].replace("\n", " "))

elapsed_seconds = time.time() - t0
print("Completed", len(RESULTS), "prompts in", elapsed_seconds, "seconds")


[1/36] another thing is how we ask the AI. we may need a AI to generate the correct input from the user input.
STATE: Ψ REASON: direct_margin_collapse DEPTH: 1 WINNER: explore 0.7325
PROFILE: input_induction
ANSWER: To resolve the issue flagged during the profile check, I will focus on addressing the specific failures identified:  1. **Compiling Not Raw**: Ensure that all generated inputs are in a format suitable for the AI system. 2. **Preserved Function**: Verify that t
[2/36] convert messy user input into the correct model-facing prompt before answering.
STATE: Ψ REASON: operational_consensus_collapse DEPTH: 1 WINNER: construct 0.745
PROFILE: input_induction
ANSWER: To convert the messy user input into the correct model-facing prompt, I need to first understand the context and requirements of the problem. However, based on the information provided in the `current_residue` object, it seems that there was an issue with the 
[3/36] design an input compiler that turns implied user inten

## 13. Save Exactly Two Output Files

In [14]:

def summarize(results: List[PromptResult]) -> Tuple[Dict[str, Any], pd.DataFrame]:
    rows = []
    for r in results:
        raw_words = word_count(r.raw_answer)
        final_words = word_count(r.answer)
        rows.append({
            "run_id": RUN_ID,
            "prompt": r.prompt,
            "state": r.state,
            "reason": r.reason,
            "depth": r.depth,
            "profile": r.final_contract.get("profile"),
            "packet_origin": r.induction_packet.get("packet_origin"),
            "packet_confidence": r.induction_packet.get("confidence"),
            "contract_count": len(r.contract_signatures),
            "winner_branch": r.winner_branch,
            "winner_origin": r.winner_origin,
            "winner_score": r.winner_score,
            "branch_count": r.metrics.get("branch_count", 0),
            "model_branch_count": r.metrics.get("model_branch_count", 0),
            "rejected_branch_count": r.metrics.get("rejected_branch_count", 0),
            "exhaust_ratio": r.metrics.get("exhaust_ratio", 0.0),
            "best_score": r.metrics.get("best_score", 0.0),
            "mean_score": r.metrics.get("mean_score", 0.0),
            "residue_count": len(r.residues),
            "shaping_accepted": r.metrics.get("shaping_accepted", False),
            "raw_words": raw_words,
            "final_words": final_words,
            "compression_ratio": 1 - final_words / max(1, raw_words),
            "answer_preview": (r.answer or "")[:260].replace("\n", " "),
        })

    df = pd.DataFrame(rows)

    aggregate = {
        "run_id": RUN_ID,
        "version": "v27",
        "purpose": "input_induction_recursive_solver",
        "model_id_or_path": MODEL_ID_OR_PATH,
        "model_ready": MODEL_READY,
        "model_generation_ready": MODEL_GENERATION_READY,
        "model_error": MODEL_ERROR,
        "smoke_text": SMOKE_TEXT,
        "dependency_status": DEPENDENCY_STATUS,
        "device_info": DEVICE_INFO,
        "config": {
            "max_depth": MAX_DEPTH,
            "branch_roles": BRANCH_ROLES,
            "max_new_tokens_branch": MAX_NEW_TOKENS_BRANCH,
            "temperature_induction": TEMPERATURE_INDUCTION,
            "temperature_branch": TEMPERATURE_BRANCH,
            "temperature_patch": TEMPERATURE_PATCH,
            "require_model_for_psi": REQUIRE_MODEL_FOR_PSI,
            "shaper_enabled": SHAPER_ENABLED,
        },
        "elapsed_seconds": elapsed_seconds,
        "total_prompts": len(results),
        "psi_count": int((df["state"] == "Ψ").sum()) if len(df) else 0,
        "omega_count": int((df["state"] == "Ω").sum()) if len(df) else 0,
        "bottom_count": int((df["state"] == "⊥").sum()) if len(df) else 0,
        "psi_ratio": float((df["state"] == "Ψ").mean()) if len(df) else 0.0,
        "mean_depth": float(df["depth"].mean()) if len(df) else 0.0,
        "mean_contract_count": float(df["contract_count"].mean()) if len(df) else 0.0,
        "mean_winner_score": float(df["winner_score"].mean()) if len(df) else 0.0,
        "mean_exhaust_ratio": float(df["exhaust_ratio"].mean()) if len(df) else 0.0,
        "mean_residue_count": float(df["residue_count"].mean()) if len(df) else 0.0,
        "reason_counts": dict(Counter(df["reason"])) if len(df) else {},
        "profile_metrics": {},
    }

    if len(df):
        for profile, g in df.groupby("profile"):
            aggregate["profile_metrics"][str(profile)] = {
                "count": int(len(g)),
                "psi_count": int((g["state"] == "Ψ").sum()),
                "omega_count": int((g["state"] == "Ω").sum()),
                "bottom_count": int((g["state"] == "⊥").sum()),
                "psi_ratio": float((g["state"] == "Ψ").mean()),
                "mean_depth": float(g["depth"].mean()),
                "mean_contract_count": float(g["contract_count"].mean()),
                "mean_winner_score": float(g["winner_score"].mean()),
                "mean_exhaust_ratio": float(g["exhaust_ratio"].mean()),
                "reason_counts": dict(Counter(g["reason"])),
            }

    return aggregate, df

aggregate, summary_df = summarize(RESULTS)

bundle = {
    "run_id": RUN_ID,
    "version": "v27",
    "purpose": "input_induction_recursive_solver",
    "aggregate": aggregate,
    "summary": summary_df.to_dict(orient="records"),
    "results": [asdict(r) for r in RESULTS],
    "profile_thresholds": PROFILE_THRESHOLDS,
    "default_semantic_locks": DEFAULT_SEMANTIC_LOCKS,
    "interpretation_lock": {
        "main_branch": "new AI runtime",
        "core_rule": "do not recurse on answers; recurse on residue by mutating the contract",
        "psi_requirement": "Ψ requires model-origin winner and polysemy-safe contract fit",
        "input_induction": "raw user input is compiled into model-facing task geometry before answer generation",
        "contract_mutation": "Ω residue produces ΔC_t patches; recursion happens through contract update",
        "not_this": ["H probing", "fallback success", "prompt text repair loop"],
    },
}

bundle_out = OUT_DIR / f"{RUN_ID}_bundle.json"
summary_out = OUT_DIR / f"{RUN_ID}_summary.csv"

with open(bundle_out, "w", encoding="utf-8") as f:
    json.dump(bundle, f, indent=2, ensure_ascii=False)

summary_df.to_csv(summary_out, index=False)

print("Saved exactly two output files:")
print(bundle_out)
print(summary_out)
print()
print(json.dumps(aggregate, indent=2, ensure_ascii=False)[:5000])
display(summary_df)


Saved exactly two output files:
D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v27_outputs\rhi_v27_3c42551ef0_bundle.json
D:\Nexus\Nexus Mark 9\NoteBooks\rhi_v27_outputs\rhi_v27_3c42551ef0_summary.csv

{
  "run_id": "rhi_v27_3c42551ef0",
  "version": "v27",
  "purpose": "input_induction_recursive_solver",
  "model_id_or_path": "Qwen/Qwen2.5-1.5B-Instruct",
  "model_ready": true,
  "model_generation_ready": true,
  "model_error": null,
  "smoke_text": "READY",
  "dependency_status": {
    "checked": true,
    "attempted_install": false,
    "missing_before": [],
    "missing_after": [],
    "errors": []
  },
  "device_info": {
    "torch_version": "2.11.0+cu126",
    "cuda_available": true,
    "device_count": 1,
    "cuda_version": "12.6",
    "gpu_name": "NVIDIA GeForce RTX 4060"
  },
  "config": {
    "max_depth": 3,
    "branch_roles": [
      "construct",
      "verify",
      "counter",
      "repair",
      "explore"
    ],
    "max_new_tokens_branch": 220,
    "temperature_induction": 0.25

,run_id,prompt,state,reason,depth,profile,packet_origin,packet_confidence,contract_count,winner_branch,...,rejected_branch_count,exhaust_ratio,best_score,mean_score,residue_count,shaping_accepted,raw_words,final_words,compression_ratio,answer_preview
0,rhi_v27_3c42551ef0,another thing is how we ask the AI. we may nee...,Ψ,direct_margin_collapse,1,input_induction,model,1.00,2,explore,...,9,0.90,0.732500,0.401480,1,False,173,173,0.000000,To resolve the issue flagged during the profil...
1,rhi_v27_3c42551ef0,convert messy user input into the correct mode...,Ψ,operational_consensus_collapse,1,input_induction,model,1.00,2,construct,...,6,0.60,0.745000,0.541430,1,False,176,176,0.000000,To convert the messy user input into the corre...
2,rhi_v27_3c42551ef0,design an input compiler that turns implied us...,Ψ,direct_margin_collapse,1,runtime_contract,model,1.00,2,counter,...,9,0.90,0.805500,0.470330,1,False,149,149,0.000000,To design an input compiler that turns implied...
3,rhi_v27_3c42551ef0,explain why the raw user prompt is not always ...,⊥,no_actionable_patch_or_max_depth,3,input_induction,model,0.95,4,construct,...,20,1.00,0.572500,0.416210,4,False,106,106,0.000000,The raw user prompt was not always the true ta...
4,rhi_v27_3c42551ef0,build a prompt coil compiler that induces the ...,⊥,no_actionable_patch_or_max_depth,3,input_induction,model,1.00,4,construct,...,20,1.00,0.697000,0.433145,4,False,123,123,0.000000,To compile a prompt for a coil compiler that i...
5,rhi_v27_3c42551ef0,how should an AI ask itself the right question...,⊥,no_actionable_patch_or_max_depth,3,input_induction,model,0.95,4,verify,...,20,1.00,0.685000,0.425980,4,False,162,162,0.000000,To formulate a clear and specific question for...
6,rhi_v27_3c42551ef0,"if its recursive it should just keep solving, ...",⊥,no_actionable_patch_or_max_depth,3,recursive_solver,model,1.00,4,verify,...,20,1.00,0.573500,0.315120,4,False,106,106,0.000000,The problem involves resolving an unresolved r...
7,rhi_v27_3c42551ef0,design a recursive AI loop that recurses on re...,⊥,no_actionable_patch_or_max_depth,3,recursive_solver,model,0.95,4,explore,...,20,1.00,0.597500,0.437175,4,False,164,164,0.000000,To design a recursive AI loop that recursively...
8,rhi_v27_3c42551ef0,explain how Ω residue becomes the next better ...,⊥,no_actionable_patch_or_max_depth,3,recursive_solver,model,1.00,4,counter,...,20,1.00,0.695000,0.551915,4,False,175,175,0.000000,The transformation of the Ω residue into the n...
9,rhi_v27_3c42551ef0,build a residue engine that mutates the contra...,⊥,no_actionable_patch_or_max_depth,3,recursive_solver,model,1.00,4,counter,...,20,1.00,0.655000,0.426810,4,False,67,67,0.000000,To mutate the contract according to the residu...


## 14. Readout

A healthy v27 run should show:

$$
\boxed{
Q_{\text{raw}}
\rightarrow
\text{InputInductionPacket}
\rightarrow
C_0
\rightarrow
\Omega_t
\rightarrow
\Delta C_t
\rightarrow
C_{t+1}
}
$$

Important diagnostics:

- `packet_origin`: should mostly be `model`; deterministic fallback is acceptable only for packet repair, not $\Psi$.
- `contract_count`: greater than 1 means residue caused contract mutation.
- `winner_origin`: $\Psi$ requires `model`.
- `residue_count`: shows whether recursion is solving actual residue.
- `reason`: distinguishes direct margin, operational consensus, and unresolved residue.

The clean target is not 100% $\Psi$.

The clean target is:

$$
\boxed{
\Psi \text{ where lawful, } \Omega \text{ where residue remains, } \bot \text{ where branch is dead.}
}
$$
